In [1]:
import os
from google.colab import files

print("Please upload your kaggle.json file...")
# This will prompt an upload button right in the notebook
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    # Safely create the .kaggle directory [cite: 179]
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

    # Move the file and set secure permissions [cite: 180, 181]
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

    # Clean up the local directory footprint
    os.remove('kaggle.json')
    print("✅ Kaggle API successfully configured and secured!")
else:
    print("❌ Error: kaggle.json was not uploaded. Please try again.")

Please upload your kaggle.json file...


Saving kaggle.json to kaggle.json
✅ Kaggle API successfully configured and secured!


In [2]:
import os

# Define our clean, production-ready directory structure
DATASET_DIR = '/content/datasets'
os.makedirs(DATASET_DIR, exist_ok=True)

print("🚀 Initiating nuclear download sequence for DeepShield datasets...")

# 1. Image Dataset: xhlulu/140k-real-and-fake-faces
print("\n📦 Downloading Image Dataset (~2.1 GB)...")
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p {DATASET_DIR}
print("📂 Unzipping Image Dataset...")
!unzip -q {DATASET_DIR}/140k-real-and-fake-faces.zip -d {DATASET_DIR}/image_data
!rm {DATASET_DIR}/140k-real-and-fake-faces.zip # Clean up zip to save memory

# 2. Audio Dataset: birdy654/deep-voice-deepfake-voice-recognition
print("\n📦 Downloading Audio Dataset (~1.5 GB)...")
!kaggle datasets download -d birdy654/deep-voice-deepfake-voice-recognition -p {DATASET_DIR}
print("📂 Unzipping Audio Dataset...")
!unzip -q {DATASET_DIR}/deep-voice-deepfake-voice-recognition.zip -d {DATASET_DIR}/audio_data
!rm {DATASET_DIR}/deep-voice-deepfake-voice-recognition.zip

print("\n✅ BOTH datasets downloaded, extracted, and cleaned up successfully!")

🚀 Initiating nuclear download sequence for DeepShield datasets...

📦 Downloading Image Dataset (~2.1 GB)...
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:32<00:00, 126MB/s]

📂 Unzipping Image Dataset...

📦 Downloading Audio Dataset (~1.5 GB)...
Dataset URL: https://www.kaggle.com/datasets/birdy654/deep-voice-deepfake-voice-recognition
License(s): other
100% 3.69G/3.69G [00:42<00:00, 92.2MB/s]

📂 Unzipping Audio Dataset...

✅ BOTH datasets downloaded, extracted, and cleaned up successfully!


In [3]:
import os
import pandas as pd
from PIL import Image
from pathlib import Path

# Define the extraction path from Step 2
IMAGE_DIR = Path('/content/datasets/image_data')

print("🔍 Initiating Nuclear Image Dataset Exploration...\n")

# Define expected structure
splits = ['train', 'valid', 'test']
classes = ['real', 'fake']
data_summary = []

# Autodetect the actual dataset root inside the unzipped folder
dataset_root = None
for path in IMAGE_DIR.rglob('train'):
    if path.is_dir():
        dataset_root = path.parent
        break

if dataset_root:
    print(f"📂 Dataset Root Identified: {dataset_root}\n")

    # Calculate counts for each split and class
    for split in splits:
        for cls in classes:
            dir_path = dataset_root / split / cls
            if dir_path.exists():
                count = len(list(dir_path.glob('*.jpg')))
                data_summary.append({'Split': split.capitalize(), 'Class': cls.capitalize(), 'Count': count})

    # Display clean, tabular results
    df = pd.DataFrame(data_summary)
    print("📊 Dataset Splits and Counts:")
    print("-" * 35)
    print(df.to_string(index=False))
    print("-" * 35)
    print(f"🎯 Total Images Verified: {df['Count'].sum()}\n")

    # Verify dimensions (Target: 256x256)
    sample_images = list(dataset_root.rglob('*.jpg'))
    if sample_images:
        sample_path = sample_images[0]
        with Image.open(sample_path) as img:
            print(f"📏 Sample Image Dimensions: {img.size[0]}x{img.size[1]} (Width x Height)")
            print(f"🖼️ Image Format/Mode: {img.format} / {img.mode}")

            if img.size == (256, 256):
                print("✅ Dimension check PASSED! Images are perfectly sized for DeepShield.")
            else:
                print("⚠️ Warning: Image dimensions are not 256x256.")
else:
    print("❌ Error: Standard train/valid/test structure not found. Please verify the extraction path.")

🔍 Initiating Nuclear Image Dataset Exploration...

📂 Dataset Root Identified: /content/datasets/image_data/real_vs_fake/real-vs-fake

📊 Dataset Splits and Counts:
-----------------------------------
Split Class  Count
Train  Real  50000
Train  Fake  50000
Valid  Real  10000
Valid  Fake  10000
 Test  Real  10000
 Test  Fake  10000
-----------------------------------
🎯 Total Images Verified: 140000

📏 Sample Image Dimensions: 256x256 (Width x Height)
🖼️ Image Format/Mode: JPEG / RGB
✅ Dimension check PASSED! Images are perfectly sized for DeepShield.


In [4]:
import os
import librosa
from pathlib import Path

AUDIO_DIR = Path('/content/datasets/audio_data')
print("🔍 Initiating Nuclear Audio Dataset Exploration...\n")

# Find all .wav files in the audio directory
wav_files = list(AUDIO_DIR.rglob('*.wav'))

if wav_files:
    print(f"🎯 Total Audio Files Found: {len(wav_files)}\n")

    # Dynamically count files by their parent directory (which acts as the class label)
    class_counts = {}
    sample_files = []

    for wav in wav_files:
        parent = wav.parent.name.lower()
        class_counts[parent] = class_counts.get(parent, 0) + 1
        # Grab a couple of samples for deep inspection
        if len(sample_files) < 2:
            sample_files.append(wav)

    # Display clean tabular results
    print("📊 Audio Dataset Counts:")
    print("-" * 35)
    for cls, count in class_counts.items():
        print(f"Class: {cls.capitalize()} | Count: {count}")
    print("-" * 35, "\n")

    # Verify Specifications (Target: 16kHz, ~3 seconds)
    print("🔬 Analyzing Audio Specifications...")
    for sample_path in sample_files:
        # Load native sample rate to verify it matches the 16kHz target
        y, sr = librosa.load(sample_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)

        print(f"File: {sample_path.name}")
        print(f" ┣━ Native Sample Rate: {sr} Hz")
        print(f" ┗━ Duration: {duration:.2f} seconds")

    if sr == 16000:
        print("\n✅ Sample rate check PASSED! Native 16kHz verified.")
    else:
        print(f"\n⚠️ Note: Native sample rate is {sr}Hz. Resampling will be required in Phase 4.")

else:
    print("❌ Error: No .wav files found. Please check the extraction path.")

🔍 Initiating Nuclear Audio Dataset Exploration...

🎯 Total Audio Files Found: 64

📊 Audio Dataset Counts:
-----------------------------------
Class: Real | Count: 8
Class: Fake | Count: 56
----------------------------------- 

🔬 Analyzing Audio Specifications...
File: linus-original.wav
 ┣━ Native Sample Rate: 44100 Hz
 ┗━ Duration: 570.36 seconds
File: trump-original.wav
 ┣━ Native Sample Rate: 48000 Hz
 ┗━ Duration: 600.43 seconds

⚠️ Note: Native sample rate is 48000Hz. Resampling will be required in Phase 4.
